# Évaluation de performance des portefeuilles BL-LLM

Notebook allégé pour comparer les portefeuilles Black-Litterman pertinents.

Le notebook est supposé être exécuté depuis le dossier `analyses/`, avec `data/` et `results/` au niveau parent.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# ============================
# Paramètres
# ============================
PROJECT_ROOT = Path("..").resolve()

START = "2015-01-01"
END = "2025-06-30"

# Ces dates doivent correspondre aux noms des fichiers générés par returns_from_weights.py
RETURNS_FILE_START = "2015-01-01"
RETURNS_FILE_END = "2025-06-30"

TAU = 0.001
PORTFOLIO_RISK_AVERSION = 1
LAMBDA_TURNOVER = 1.0

MODELS = {
    "gpt54mini": "BL-GPT-5.4-mini",
    "gemma3": "BL-Gemma3",
    "qwen": "BL-Qwen",
    "llama": "BL-Llama",
}

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / f"portfolio_risk_aversion_{PORTFOLIO_RISK_AVERSION:g}"

if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"Dossier introuvable: {RESULTS_DIR}")

print(f"Dossier de résultats: {RESULTS_DIR}")
print(f"Période: {START} -> {END} | tau={TAU} | lambda={PORTFOLIO_RISK_AVERSION:g}")


## Chargement des rendements et des poids BL

In [ ]:
def read_returns(path: Path, name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "Portfolio_Return" not in df.columns:
        raise ValueError(f"{name}: colonne Portfolio_Return manquante dans {path.name}")
    return df


def read_weights(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "Date" not in df.columns:
        raise ValueError(f"Colonne Date manquante dans {path.name}")
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    return df.dropna(subset=["Date"]).set_index("Date").sort_index()


def bl_paths(model: str) -> tuple[Path, Path]:
    base = f"{model}_base_omega_1"
    weights_path = RESULTS_DIR / f"{base}_black_litterman_market_implied_weights_tau_{TAU}.csv"
    returns_path = RESULTS_DIR / (
        f"{base}_black_litterman_implied_weights_returns_tau_"
        f"{TAU}_{RETURNS_FILE_START}_{RETURNS_FILE_END}.csv"
    )
    return weights_path, returns_path


def none_paths() -> tuple[Path, Path]:
    weights_path = RESULTS_DIR / f"none_black_litterman_market_implied_weights_tau_{TAU}.csv"
    returns_path = RESULTS_DIR / (
        f"none_black_litterman_implied_weights_returns_tau_"
        f"{TAU}_{RETURNS_FILE_START}_{RETURNS_FILE_END}.csv"
    )
    return weights_path, returns_path


returns_raw = {}
weights_raw = {}

# BL sans vues
none_w_path, none_r_path = none_paths()
if none_r_path.exists():
    returns_raw["BL-None"] = read_returns(none_r_path, "BL-None")
if none_w_path.exists():
    weights_raw["BL-None"] = read_weights(none_w_path)

# Modèles LLM
for model, label in MODELS.items():
    w_path, r_path = bl_paths(model)

    if r_path.exists():
        returns_raw[label] = read_returns(r_path, label)
    else:
        print(f"Rendements manquants pour {label}: {r_path.name}")

    if w_path.exists():
        weights_raw[label] = read_weights(w_path)
    else:
        print(f"Poids manquants pour {label}: {w_path.name}")

print("Portefeuilles chargés:", list(returns_raw))


## Benchmark S&P 25 cap-weighted

In [ ]:
def build_sp25_cap_weighted_returns(
    csv_path: Path = DATA_DIR / "filtered_sp25_data.csv",
    start: str = START,
    end: str = END,
) -> pd.DataFrame:
    df = pd.read_csv(csv_path, low_memory=False)

    required = {"tic", "stock_ret", "market_equity"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Colonnes manquantes dans {csv_path.name}: {sorted(missing)}")

    if "date" in df.columns:
        raw_date = df["date"].astype(str).str.strip()
        df["date_key"] = pd.to_datetime(raw_date, format="%Y%m%d", errors="coerce")
        missing_dates = df["date_key"].isna()
        df.loc[missing_dates, "date_key"] = pd.to_datetime(raw_date[missing_dates], errors="coerce")
    elif {"year", "month"}.issubset(df.columns):
        df["date_key"] = pd.to_datetime(
            {
                "year": pd.to_numeric(df["year"], errors="coerce"),
                "month": pd.to_numeric(df["month"], errors="coerce"),
                "day": 1,
            },
            errors="coerce",
        )
    else:
        raise ValueError("Le fichier doit contenir soit date, soit year/month.")

    df = df.dropna(subset=["date_key", "tic"]).copy()
    df["tic"] = df["tic"].astype(str).str.upper().str.strip().str.replace("-", ".", regex=False)
    df["stock_ret"] = pd.to_numeric(df["stock_ret"], errors="coerce")
    df["market_equity"] = pd.to_numeric(df["market_equity"], errors="coerce")
    df["ym"] = df["date_key"].dt.to_period("M")

    monthly = (
        df.sort_values(["ym", "tic", "date_key"])
          .drop_duplicates(["ym", "tic"], keep="last")
    )

    returns = monthly.pivot(index="ym", columns="tic", values="stock_ret").sort_index()
    caps = monthly.pivot(index="ym", columns="tic", values="market_equity").sort_index().shift(1)

    rows = []
    previous_w = None

    for ym in returns.index:
        r = returns.loc[ym]
        c = caps.loc[ym]
        valid = r.notna() & c.notna() & (c > 0)

        if valid.sum() < 2:
            continue

        w = c[valid] / c[valid].sum()
        port_ret = float((w * r[valid]).sum())

        w_full = w.reindex(returns.columns, fill_value=0.0)
        turnover = 0.0 if previous_w is None else float((w_full - previous_w).abs().sum())

        rows.append({
            "date_key": ym.to_timestamp(),
            "Portfolio_Return": port_ret,
            "Turnover": turnover,
            "n_assets": int(valid.sum()),
        })

        previous_w = w_full

    out = pd.DataFrame(rows)
    out = out[(out["date_key"] >= pd.to_datetime(start)) & (out["date_key"] <= pd.to_datetime(end))]
    return out.reset_index(drop=True)


sp25_returns = build_sp25_cap_weighted_returns()
returns_raw = {"S&P 25": sp25_returns, **returns_raw}

print(f"S&P 25: {sp25_returns['date_key'].min().date()} -> {sp25_returns['date_key'].max().date()} | {len(sp25_returns)} mois")


## Alignement des séries mensuelles

In [ ]:
def to_monthly_return_series(df: pd.DataFrame, name: str) -> pd.Series:
    if "date_key" in df.columns:
        dates = pd.to_datetime(df["date_key"], errors="coerce")
    elif "Date" in df.columns:
        dates = pd.to_datetime(df["Date"], errors="coerce")
    else:
        raise ValueError(f"{name}: colonne de date introuvable.")

    s = pd.Series(pd.to_numeric(df["Portfolio_Return"], errors="coerce").values, index=dates, name=name)
    s = s.dropna()
    s.index = s.index.to_period("M").to_timestamp()
    return s[~s.index.duplicated(keep="last")].sort_index()


def to_monthly_column_series(df: pd.DataFrame, col: str, name: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series(dtype=float, name=name)

    if "date_key" in df.columns:
        dates = pd.to_datetime(df["date_key"], errors="coerce")
    elif "Date" in df.columns:
        dates = pd.to_datetime(df["Date"], errors="coerce")
    else:
        return pd.Series(dtype=float, name=name)

    s = pd.Series(pd.to_numeric(df[col], errors="coerce").values, index=dates, name=name).dropna()
    s.index = s.index.to_period("M").to_timestamp()
    return s[~s.index.duplicated(keep="last")].sort_index()


returns = {name: to_monthly_return_series(df, name) for name, df in returns_raw.items()}
turnovers = {name: to_monthly_column_series(df, "Turnover", name) for name, df in returns_raw.items()}

common_idx = None
for s in returns.values():
    common_idx = s.index if common_idx is None else common_idx.intersection(s.index)

common_idx = common_idx.sort_values()
returns = {name: s.loc[common_idx] for name, s in returns.items()}
turnovers = {name: s.reindex(common_idx) for name, s in turnovers.items()}

print(f"Période commune: {common_idx.min().date()} -> {common_idx.max().date()} | {len(common_idx)} mois")


## Métriques de performance

In [ ]:
def load_risk_free_monthly(csv_path: Path = DATA_DIR / "DGS1.csv") -> pd.Series:
    df = pd.read_csv(csv_path)

    date_col = next((c for c in df.columns if c.lower() in {"date", "observation_date"}), df.columns[0])
    value_col = next((c for c in df.columns if c != date_col), df.columns[1])

    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df = df.dropna(subset=[date_col]).sort_values(date_col)

    rf = df.set_index(date_col)[value_col] / 100.0
    rf = rf.resample("ME").last()
    rf.index = rf.index.to_period("M").to_timestamp()
    return rf


def performance_metrics(ret: pd.Series, benchmark: pd.Series, rf_monthly_annual: pd.Series) -> dict:
    periods_per_year = 12

    rf = rf_monthly_annual.reindex(ret.index).ffill().bfill() / periods_per_year
    excess = ret - rf
    active = ret - benchmark

    wealth = (1 + ret).cumprod()
    drawdown = wealth / wealth.cummax() - 1

    total_return = (1 + ret).prod() - 1
    years = len(ret) / periods_per_year

    vol = ret.std(ddof=1) * np.sqrt(periods_per_year)
    tracking_error = active.std(ddof=1)

    return {
        "Rend. cumulé": total_return,
        "CAGR": (1 + total_return) ** (1 / years) - 1,
        "Vol. ann.": vol,
        "Sharpe": (excess.mean() / ret.std(ddof=1)) * np.sqrt(periods_per_year) if ret.std(ddof=1) > 0 else np.nan,
        "IR": (active.mean() / tracking_error) * np.sqrt(periods_per_year) if tracking_error > 0 else np.nan,
        "Max DD": drawdown.min(),
    }


rf = load_risk_free_monthly()
benchmark = returns["S&P 25"]

stats = {}
for name, ret in returns.items():
    m = performance_metrics(ret, benchmark, rf)
    avg_turnover = turnovers.get(name, pd.Series(dtype=float)).mean(skipna=True)
    m["Turnover mensuel moyen"] = avg_turnover
    m["Sharpe ajusté turnover"] = (
        m["Sharpe"] - LAMBDA_TURNOVER * avg_turnover
        if pd.notna(m["Sharpe"]) and pd.notna(avg_turnover)
        else np.nan
    )
    stats[name] = m

stats_df = pd.DataFrame(stats).T
stats_df


## Rendement cumulatif

In [ ]:
plt.figure(figsize=(12, 7))

for name, ret in returns.items():
    wealth = (1 + ret).cumprod()
    initial_date = wealth.index[0] - pd.offsets.MonthBegin(1)
    wealth = pd.concat([pd.Series([1.0], index=[initial_date]), wealth])
    plt.plot(wealth.index, wealth.values, label=name)

plt.yscale("log")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xlabel("Date")
plt.ylabel("Indice de richesse, base 1")
plt.title(f"Rendement cumulatif des portefeuilles BL | tau={TAU} | lambda={PORTFOLIO_RISK_AVERSION:g}")
plt.legend(loc="upper left")
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()


## Turnover mensuel moyen

In [ ]:
turnover_summary = pd.Series(
    {name: s.mean(skipna=True) for name, s in turnovers.items()},
    name="Turnover mensuel moyen",
).sort_values(ascending=False)

turnover_summary.to_frame()
